<div style="background-color:#234c7d; border-radius:8px; padding:32px 40px 24px; margin-bottom:8px;">
  <h1 style="margin:0 0 6px; font-size:28px; font-weight:600; color:#C49A28; letter-spacing:-0.5px;">
    GA Image Approximation
  </h1>
  <p style="margin:0; font-size:13px; color:#a8b8d0; letter-spacing:0.5px;">
    Challenge 1 — Fitness Function Comparison: RMSE vs CIEDE2000 vs SSIM
  </p>
</div>

<div style="border:1.5px solid #234c7d; border-radius:8px; padding:20px 40px; margin-bottom:8px; display:flex; gap:48px;">
  <div>
    <p style="margin:0 0 8px; font-size:11px; font-weight:600; color:#888; letter-spacing:1px; text-transform:uppercase;">Group 35 - Convergence</p>
    <p style="margin:0; font-size:14px; color:#2d2d2d; line-height:2;">
      Alexandra Rodrigues, 20250514 &nbsp;<br>
      Francisca Fernandes, 20250406 &nbsp;<br>
      Gonçalo Arrobas, 20250421 &nbsp;<br>
      Mariana Melo, 20250414  &nbsp;<br>
    </p>
  </div>
  <div>
    <p style="margin:0 0 8px; font-size:11px; font-weight:600; color:#888; letter-spacing:1px; text-transform:uppercase;">Course</p>
    <p style="margin:0; font-size:14px; color:#2d2d2d; line-height:2;">
      Computational Intelligence for Optimization<br>
      MSc in Data Science &amp; Advanced Analytics<br>
      Nova Information Management School (NOVA IMS)<br>
      2025/2026
    </p>
  </div>
</div>

## **<span style="color:#234c7d">Table of Contents</span>**

1. [Introduction](#1-introduction)
2. [Setup and Configuration](#2-setup)
3. [Cross-Evaluation](#3-cross-evaluation)
4. [Convergence Analysis](#4-convergence-analysis)
5. [Population Dynamics](#5-population-dynamics)
6. [Statistical Validation](#6-statistical-validation)
7. [Visual Comparison](#7-visual-comparison)
8. [Evolution Snapshots](#8-evolution-snapshots)
9. [Summary and Conclusions](#9-summary)

---
# 1. Introduction

This notebook evaluates **Challenge 1**: does replacing the standard RMSE fitness function with a perceptual colour metric improve the visual quality of the GA output?

Three independent GA runs were conducted, each trained with a different fitness function, all using the best operator configuration found in Phases 1-11:

| Fitness Function | Description | Range |
|---|---|---|
| **RMSE** | Pixel-wise Root Mean Squared Error in RGB space | [0, 255] lower is better |
| **CIEDE2000** | Perceptual colour difference in CIELAB space | [0, inf) lower is better |
| **SSIM** | Structural Similarity Index (returned as 1 - SSIM) | [0, 2] lower is better |

Each best individual is **cross-evaluated** with all three metrics, allowing a direct comparison of trade-offs.

**Operator configuration used for all three runs:**

| Component | Choice |
|---|---|
| Selection | Tournament k=10 |
| Crossover | Blend alpha=0.5 |
| Mutation | Gaussian Decay sigma 80 to 2 |
| Elitism | 3 elites |
| Init strategy | Quadrant |
| Generations | 3000 |
| Seeds | 42, 43, 44 |

---
# 2. Setup and Configuration

In [ ]:
import json
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy import stats

sys.path.insert(0, '../src')
from fitness import RMSEFitness, CIEDEFitness, SSIMFitness
from utils import load_target, render, triangles_from_json

BLUE  = '#002654'
GOLD  = '#C49A28'
COLORS = {'RMSE': '#002654', 'CIEDE2000': '#C49A28', 'SSIM': '#4C72B0'}

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

os.makedirs('../outputs', exist_ok=True)
print('Setup complete.')

In [ ]:
RUNNER_OUTPUTS = '../runner_outputs'
TARGET_PATH    = '../data/Girl_Pearl_Earing.png'
SEEDS          = [42, 43, 44]

RUNS = {
    'RMSE':      'p10_rate_001_sigmax_80',
    'CIEDE2000': 'p13_fitness_ciede2000/p13_fitness_ciede2000',
    'SSIM':      'p13_fitness_ssim/p13_fitness_ssim',
}

def load_generation_log(run_folder, seed):
    path = os.path.join(RUNNER_OUTPUTS, run_folder, f'seed_{seed}', 'generation_log.json')
    if not os.path.exists(path):
        print(f'  WARNING: not found -- {path}')
        return None
    with open(path) as f:
        return json.load(f)

def get_curves(metric_name, key):
    curves = []
    for seed in SEEDS:
        log = load_generation_log(RUNS[metric_name], seed)
        if log:
            curves.append([e.get(key, float('nan')) for e in log])
    if not curves:
        return None, None, None
    min_len = min(len(c) for c in curves)
    arr = np.array([c[:min_len] for c in curves])
    return np.arange(min_len), arr.mean(axis=0), arr.std(axis=0)

print('Verifying generation logs:')
for metric, folder in RUNS.items():
    for seed in SEEDS:
        log = load_generation_log(folder, seed)
        status = f'{len(log)} generations' if log else 'NOT FOUND'
        print(f'  {metric} seed={seed}: {status}')

---
# 3. Cross-Evaluation

To fairly compare the three fitness functions, each best individual — regardless of which metric it was trained with — is evaluated against **all three metrics simultaneously**. This cross-evaluation design is essential: comparing a CIEDE-trained individual only by its CIEDE score would be circular. Instead, we ask a more meaningful question: does training with a perceptual metric produce an individual that is also better by other standards?

The table and bar charts below show mean ± std across the 3 seeds. The gold bar in each chart highlights the GA whose training metric matches the evaluation metric.

In [ ]:
target   = load_target(TARGET_PATH)
rmse_fn  = RMSEFitness(target)
ciede_fn = CIEDEFitness(target)
ssim_fn  = SSIMFitness(target)

cross_results = []
for metric_name, run_folder in RUNS.items():
    for seed in SEEDS:
        tpath = os.path.join(RUNNER_OUTPUTS, run_folder, f'seed_{seed}', 'best_final_triangles.json')
        if not os.path.exists(tpath):
            print(f'  WARNING: {tpath}')
            continue
        rendered = render(triangles_from_json(tpath))
        cross_results.append({
            'trained_with': metric_name,
            'seed':         seed,
            'rmse':         rmse_fn.evaluate(rendered),
            'ciede2000':    ciede_fn.evaluate(rendered),
            'ssim':         1.0 - ssim_fn.evaluate(rendered),
        })

df_cross = pd.DataFrame(cross_results)

summary = df_cross.groupby('trained_with').agg(
    rmse_mean=('rmse','mean'), rmse_std=('rmse','std'),
    ciede_mean=('ciede2000','mean'), ciede_std=('ciede2000','std'),
    ssim_mean=('ssim','mean'), ssim_std=('ssim','std'),
).round(4)

print(f"{'Trained with':<14} {'RMSE':>18} {'CIEDE2000':>18} {'SSIM':>18}")
print('─' * 72)
for name, row in summary.iterrows():
    print(f"{name:<14} {row['rmse_mean']:>8.4f} +/- {row['rmse_std']:>6.4f}   "
          f"{row['ciede_mean']:>8.4f} +/- {row['ciede_std']:>6.4f}   "
          f"{row['ssim_mean']:>8.4f} +/- {row['ssim_std']:>6.4f}")

In [ ]:
metrics_plot = [('rmse','RMSE'), ('ciede2000','CIEDE2000 (dE00)'), ('ssim','SSIM (similarity)')]
x = np.arange(len(RUNS))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (metric_col, metric_label) in zip(axes, metrics_plot):
    means = [df_cross[df_cross['trained_with'] == m][metric_col].mean() for m in RUNS]
    stds  = [df_cross[df_cross['trained_with'] == m][metric_col].std()  for m in RUNS]
    bests = [df_cross[df_cross['trained_with'] == m][metric_col].min()  for m in RUNS]
    bars = ax.bar(x, means, yerr=stds, capsize=4, color=BLUE, alpha=0.75, edgecolor='white', width=0.5)
    for i, trained in enumerate(RUNS):
        if trained.lower().replace('2000','') in metric_col.lower().replace('2000',''):
            bars[i].set_color(GOLD)
            bars[i].set_alpha(0.9)
    ax.scatter(x, bests, color=GOLD, zorder=5, s=60, label='best seed')
    ax.set_xticks(x)
    ax.set_xticklabels(list(RUNS.keys()), rotation=15, ha='right', fontsize=9)
    ax.set_ylabel(metric_label)
    ax.set_title(f'Evaluated with {metric_label}', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Cross-Evaluation — each GA trained on one metric, evaluated on all three',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/challenge1_cross_eval.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 4. Convergence Analysis

Convergence curves show how the best fitness and mean population fitness evolve over 3000 generations. Shaded bands represent ± 1 standard deviation across the 3 seeds — wider bands indicate more variability between seeds.

Note that the three GAs optimise different objectives, so their fitness scales are not directly comparable. What matters here is the **shape** of the curves: how fast each GA converges, whether it plateaus early, and how stable convergence is across seeds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for key, ylabel, title, ax in [
    ('best', 'Best fitness (training metric)', 'Best fitness per GA', axes[0]),
    ('mean', 'Mean population fitness',        'Mean population fitness', axes[1]),
]:
    for metric_name, color in COLORS.items():
        gens, mean, std = get_curves(metric_name, key)
        if gens is None:
            continue
        ax.plot(gens, mean, color=color, lw=2, label=metric_name)
        ax.fill_between(gens, mean - std, mean + std, alpha=0.12, color=color)
    ax.set_xlabel('Generation')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Convergence Analysis -- RMSE vs CIEDE2000 vs SSIM', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/challenge1_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 5. Population Dynamics

Two complementary views of how each population behaves internally over generations.

**Diversity (phenotypic variance)** measures how spread out the fitness values are within the population. High diversity means the population is still exploring; low diversity means it has converged to a narrow region of the search space. A premature collapse in diversity is a warning sign of premature convergence.

**Population spread (mean − best)** measures the gap between the average individual and the best individual. A large gap means the population is heterogeneous and still evolving; a gap near zero means the population has clustered around the best solution found.

Together these two views help explain *why* a particular GA converged well or poorly — not just *that* it did.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for metric_name, color in COLORS.items():
    gens, mean, std = get_curves(metric_name, 'diversity')
    if gens is None: continue
    ax.plot(gens, mean, color=color, lw=2, label=metric_name)
    ax.fill_between(gens, mean - std, mean + std, alpha=0.12, color=color)
ax.set_xlabel('Generation')
ax.set_ylabel('Phenotypic variance')
ax.set_title('Population diversity over generations', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
for metric_name, color in COLORS.items():
    _, mean_b, _ = get_curves(metric_name, 'best')
    _, mean_m, _ = get_curves(metric_name, 'mean')
    if mean_b is None or mean_m is None: continue
    n = min(len(mean_b), len(mean_m))
    ax.plot(np.arange(n), mean_m[:n] - mean_b[:n], color=color, lw=2, label=metric_name)
ax.set_xlabel('Generation')
ax.set_ylabel('Mean - Best fitness')
ax.set_title('Population spread (mean - best)\nSmaller = population converging', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Population Dynamics -- RMSE vs CIEDE2000 vs SSIM', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/challenge1_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 6. Statistical Validation

## 6.1 Motivation

Genetic Algorithms are stochastic — the same configuration run twice with different seeds will produce different results. With only 3 seeds per configuration, observed differences in cross-evaluation scores could reflect genuine performance differences or simply random variation. Statistical tests allow us to distinguish between the two.

We follow the methodology recommended by Derrac et al. (2011) for comparing evolutionary algorithms: non-parametric tests that make no assumptions about the distribution of results.

## 6.2 Pipeline

The testing pipeline has three steps:

1. **Shapiro-Wilk** — checks whether the 3 fitness values per configuration are consistent with a normal distribution. With n=3 this test has low power, but we include it for completeness.
2. **Mann-Whitney U** — non-parametric pairwise comparison between two groups. Does not assume normality and is appropriate for small samples.
3. **Rank-biserial r** — effect size measuring the magnitude of the difference, independent of sample size. Values below 0.1 are negligible, above 0.5 are large.

Significance threshold: α = 0.05.

> Derrac, J., García, S., Molina, D., & Herrera, F. (2011). A practical tutorial on the use of nonparametric statistical tests as a methodology for comparing evolutionary and swarm intelligence algorithms. *Swarm and Evolutionary Computation*, 1(1), 3–18.

In [ ]:
def rank_biserial_r(a, b):
    u, _ = stats.mannwhitneyu(a, b, alternative='two-sided')
    return 1 - (2 * u) / (len(a) * len(b))

def effect_label(r):
    r = abs(r)
    if r < 0.1: return 'negligible'
    if r < 0.3: return 'small'
    if r < 0.5: return 'medium'
    return 'large'

pairs        = [('RMSE','CIEDE2000'), ('RMSE','SSIM'), ('CIEDE2000','SSIM')]
metrics_stat = [('rmse','RMSE'), ('ciede2000','CIEDE2000'), ('ssim','SSIM (similarity)')]

print('=' * 70)
print('STATISTICAL TESTS -- Mann-Whitney U (alpha = 0.05)')
print('=' * 70)

for name_a, name_b in pairs:
    print(f'\n  {name_a} vs {name_b}')
    for metric_col, metric_label in metrics_stat:
        a = df_cross[df_cross['trained_with'] == name_a][metric_col].values
        b = df_cross[df_cross['trained_with'] == name_b][metric_col].values
        _, p_sw_a = stats.shapiro(a)
        _, p_sw_b = stats.shapiro(b)
        _, p_mw   = stats.mannwhitneyu(a, b, alternative='two-sided')
        r         = rank_biserial_r(a, b)
        sig = 'significant' if p_mw < 0.05 else 'not significant'
        print(f'    [{metric_label}]')
        print(f'      {name_a}: mean={np.mean(a):.4f}  std={np.std(a):.4f}  (Shapiro p={p_sw_a:.3f})')
        print(f'      {name_b}: mean={np.mean(b):.4f}  std={np.std(b):.4f}  (Shapiro p={p_sw_b:.3f})')
        print(f'      Mann-Whitney U: p={p_mw:.4f} -> {sig}')
        print(f'      Effect size: r={r:.3f} ({effect_label(r)})')

---
# 7. Visual Comparison

The ultimate test of any image approximation algorithm is visual. Numerical metrics capture important properties of image quality but cannot fully replace human judgement — particularly for a painterly reconstruction task where perceptual fidelity matters more than pixel-level accuracy.

Each panel shows the best result (by its own training metric) for one fitness function, with all three cross-evaluation scores shown below. This allows a direct visual assessment of whether the numerical differences in Section 3 translate into visible differences in image quality.

In [ ]:
target_img   = mpimg.imread(TARGET_PATH)
metric_cols  = {'RMSE': 'rmse', 'CIEDE2000': 'ciede2000', 'SSIM': 'ssim'}

fig, axes = plt.subplots(1, 4, figsize=(22, 7))
axes[0].imshow(target_img)
axes[0].set_title('Original\nVermeer -- Girl with a Pearl Earring', fontsize=10)
axes[0].axis('off')

for ax, metric_name in zip(axes[1:], RUNS):
    subset = df_cross[df_cross['trained_with'] == metric_name]
    col    = metric_cols[metric_name]
    best_seed = int(
        subset.loc[subset[col].idxmax(), 'seed'] if metric_name == 'SSIM'
        else subset.loc[subset[col].idxmin(), 'seed']
    )
    img_path = os.path.join(RUNNER_OUTPUTS, RUNS[metric_name], f'seed_{best_seed}', 'best_final.png')
    if not os.path.exists(img_path):
        ax.text(0.5, 0.5, 'Image not found', ha='center', va='center')
        ax.axis('off')
        continue
    row = subset[subset['seed'] == best_seed].iloc[0]
    ax.imshow(mpimg.imread(img_path))
    ax.set_title(
        f'Trained: {metric_name}  (seed {best_seed})\n'
        f'RMSE={row["rmse"]:.3f}  |  dE={row["ciede2000"]:.3f}  |  SSIM={row["ssim"]:.3f}',
        fontsize=9
    )
    ax.axis('off')

plt.suptitle('Visual Comparison -- Best Result per Fitness Function', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/challenge1_visual.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 8. Evolution Snapshots

Checkpoint images at fixed generation intervals (seed 42) show how each GA's best individual evolves visually over time. This gives an intuition for the different learning dynamics produced by each fitness function — for example, whether CIEDE2000 develops colour fidelity earlier than RMSE, or whether SSIM converges to a structurally coherent but chromatically incorrect result from the start.

In [ ]:
CHECKPOINT_GENS = [0, 500, 1000, 1500, 2000, 2500, 3000]
REF_SEED        = 42

fig, axes = plt.subplots(len(RUNS), len(CHECKPOINT_GENS),
                         figsize=(len(CHECKPOINT_GENS) * 2.8, len(RUNS) * 3.5))

for row_idx, (metric_name, run_folder) in enumerate(RUNS.items()):
    for col_idx, gen in enumerate(CHECKPOINT_GENS):
        ax = axes[row_idx, col_idx]
        img_path = os.path.join(RUNNER_OUTPUTS, run_folder,
                                f'seed_{REF_SEED}', 'checkpoints', f'gen_{gen:04d}.png')
        if os.path.exists(img_path):
            ax.imshow(mpimg.imread(img_path))
        else:
            ax.text(0.5, 0.5, f'gen {gen}\nnot found',
                    ha='center', va='center', transform=ax.transAxes, fontsize=8, color='#888')
        ax.axis('off')
        if row_idx == 0:
            ax.set_title(f'Gen {gen}', fontsize=9)
        if col_idx == 0:
            ax.text(-0.12, 0.5, metric_name, transform=ax.transAxes,
                    fontsize=11, fontweight='bold', va='center', ha='right',
                    rotation=90, color=BLUE)

plt.suptitle(f'Evolution Snapshots (seed {REF_SEED}) -- RMSE vs CIEDE2000 vs SSIM',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/challenge1_evolution.png', dpi=130, bbox_inches='tight')
plt.show()

---
# 9. Summary and Conclusions

In [ ]:
best_rmse_ga  = df_cross.groupby('trained_with')['rmse'].mean().idxmin()
best_ciede_ga = df_cross.groupby('trained_with')['ciede2000'].mean().idxmin()
best_ssim_ga  = df_cross.groupby('trained_with')['ssim'].mean().idxmax()

print('─' * 60)
print('CHALLENGE 1 -- SUMMARY')
print('─' * 60)
print(f'  Best RMSE score:      GA trained with {best_rmse_ga}')
print(f'  Best CIEDE2000 score: GA trained with {best_ciede_ga}')
print(f'  Best SSIM score:      GA trained with {best_ssim_ga}')
print()
print('Cross-evaluation (mean +/- std, 3 seeds):')
print(f"{'Trained with':<14} {'RMSE':>18} {'CIEDE2000':>18} {'SSIM':>18}")
print('─' * 72)
for name, row in summary.iterrows():
    print(f"{name:<14} {row['rmse_mean']:>8.4f} +/- {row['rmse_std']:>6.4f}   "
          f"{row['ciede_mean']:>8.4f} +/- {row['ciede_std']:>6.4f}   "
          f"{row['ssim_mean']:>8.4f} +/- {row['ssim_std']:>6.4f}")

## **<span style="color:#002654">Key Findings</span>**

> *To be completed after results analysis.*

| Finding | Evidence |
|---------|----------|
| CIEDE2000 GA achieves lower perceptual error | Cross-evaluation table, Section 3 |
| RMSE GA achieves lower pixel error | Cross-evaluation table, Section 3 |
| SSIM fails as a training metric | Visual comparison Section 7; convergence Section 4 |
| Fitness choice shapes what the GA learns to optimise | Statistical tests Section 6 |

**On SSIM:** Training with SSIM produced visually incorrect results with wrong colours. This is consistent with the known limitation of SSIM as an optimisation objective for image synthesis: when the generated image is structurally dissimilar from the target (as it is in early generations), the SSIM loss surface is nearly flat and provides insufficient signal for selection. Wang et al. (2004) note that SSIM is most suitable for assessing quality of images that are already similar to the reference.

**On CIEDE2000:** Training with a perceptual metric produces qualitatively different results. Colours in semantically important regions (skin tones, the blue headscarf) appear more faithful. This supports the hypothesis that RMSE misallocates optimisation effort to low-salience regions such as the dark background, while CIEDE2000 implicitly focuses on regions where colour errors are most perceptible to the human eye.

---
*References*

*Sharma, G., Wu, W., & Dalal, E. N. (2005). The CIEDE2000 color-difference formula. Color Research & Application, 30(1), 21-30.*

*Wang, Z., Bovik, A. C., Sheikh, H. R., & Simoncelli, E. P. (2004). Image quality assessment: from error visibility to structural similarity. IEEE Transactions on Image Processing, 13(4), 600-612.*

*Derrac, J., Garcia, S., Molina, D., & Herrera, F. (2011). A practical tutorial on the use of nonparametric statistical tests. Swarm and Evolutionary Computation, 1(1), 3-18.*